Read Bronze as a stream

In [0]:
cdc_stream = (
    spark.readStream
    .table("cdc_catalog.bronze.bronze_customer_cdc")
    .withWatermark("updated_at", "10 minutes")
)


Apply the event-time watermark

Create the Delta target

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

target=DeltaTable.forName(spark,'cdc_catalog.silver.silver_customer')

Create the CDC processing function

In [0]:
def process_cdc_batch(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    latest_batch = (
        batch_df
        .orderBy(col("updated_at").desc())
        .dropDuplicates(["customer_id"])
    )

    (
        target.alias("t")
        .merge(
            latest_batch.alias("s"),
            "t.customer_id = s.customer_id"
        )

        # UPDATE
        .whenMatchedUpdate(
            condition="""
                s.operation = 'U'
                AND s.updated_at > t.updated_at
            """,
            set={
                "name": "s.name",
                "city": "s.city",
                "salary": "s.salary",
                "updated_at": "s.updated_at"
            }
        )

        # DELETE
        .whenMatchedDelete(
            condition="""
                s.operation = 'D'
                AND s.updated_at > t.updated_at
            """
        )

        # INSERT
        .whenNotMatchedInsert(
            condition="s.operation = 'I'",
            values={
                "customer_id": "s.customer_id",
                "name": "s.name",
                "city": "s.city",
                "salary": "s.salary",
                "updated_at": "s.updated_at"
            }
        )

        .execute()
    )


start the CDC stream

In [0]:
query = (
    cdc_stream
    .writeStream
    .foreachBatch(process_cdc_batch)
    .option(
        "checkpointLocation",
        "/Volumes/cdc_catalog/cdc_demo/cdc_files/_checkpoint/cdc_merge"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
